# No Plan but Everything Under Control: Minimal 2D Path Sequencing
This notebook provides a minimal mathematical proof-of-concept for the AICON framework (Dynamically Composed Gradient Descent).

### Scenario: The Locked Door
- An agent starts at `(0, 0)` and needs to reach a goal at `(10, 0)`.
- A barrier (gate) exists at $X=5$.
- A button exists at `(2, 5)`.
- **Active Interconnection:** The barrier's opening probability $c_{gate}$ is a differentiable function of the agent's distance to the button.

AICON implicitly sequences the subgoals (*go to button*, then *go through gate to goal*) by following the steepest available gradient in the composed cost function. No explicit state-machines or path planning algorithms are used.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output

### 1. Defining the AICON Components and Interconnections
Here we define the PyTorch differentiable functions corresponding to `Eq. 1` (Components) and `Eq. 2` (Interconnections) from the paper.

In [ ]:
class PointMassAICON:
    def __init__(self, start_pos, goal_pos, button_pos, gate_x=5.0):
        self.device = torch.device('cpu')
        # The tracked state variable (requires_grad=True so we can pull actions from it)
        self.agent_pos = torch.tensor(start_pos, dtype=torch.float32, requires_grad=True, device=self.device)
        self.goal_pos = torch.tensor(goal_pos, dtype=torch.float32, device=self.device)
        self.button_pos = torch.tensor(button_pos, dtype=torch.float32, device=self.device)
        self.gate_x = gate_x
        
    def gate_open_prob(self, pos):
        """Active Interconnection (Eq 2): c_gate = f(||x - B||)"""
        dist_to_button = torch.norm(pos - self.button_pos)
        # Use an exponential backoff to yield smooth gradients
        # Returns ~1 when near button, ~0 when far
        return torch.exp(-0.5 * dist_to_button)

    def compute_cost(self, pos):
        """
        The overall differentiating goal function g(x) (Eq 3) containing all sub-costs.
        """
        # 1. Primary objective
        goal_cost = torch.norm(pos - self.goal_pos)
        
        # 2. Extract interconnection dependencies
        c_gate = self.gate_open_prob(pos)
        
        # 3. Soft constraint: It costs a LOT to cross the gate_x if the gate is closed.
        # (Uses a Gaussian barrier centered at gate_x)
        gate_distance = pos[0] - self.gate_x
        barrier_cost = 100.0 * torch.exp(-0.5 * (gate_distance / 2.0)**2)
        
        # The barrier penalty is modulated strictly by the gate interconnection
        total_cost = goal_cost + (1.0 - c_gate) * barrier_cost
        return total_cost

    def step(self, lr=0.5):
        """
        Dynamically Composed Gradient Descent (Eq 4). 
        Automatically falls into the steepest active path.
        """
        # Reset gradients cleanly
        if self.agent_pos.grad is not None:
            self.agent_pos.grad.zero_()
            
        # Forward Pass -> Propagates through Sub-estimators & Interconnections
        cost = self.compute_cost(self.agent_pos)
        
        # Backward Pass -> Enumerates all paths and returns steepest gradient
        cost.backward()
        
        # Normalize the gradient to act as a proper velocity vector
        grad = self.agent_pos.grad.clone()
        grad_norm = torch.norm(grad)
        if grad_norm > 1e-6:
            grad_normalized = grad / grad_norm
        else:
            grad_normalized = grad
            
        # Apply optimal action
        with torch.no_grad():
            self.agent_pos -= lr * grad_normalized
            
        return self.agent_pos.clone().detach().numpy(), cost.item()

### 2. Execution and Plotting
We simply loop the `step()` function. Watch how the pure mathematical gradients automatically choose to bend upwards to hit the button, and then safely proceed to the goal.

In [ ]:
aicon = PointMassAICON(start_pos=[0.0, 0.0], goal_pos=[10.0, 0.0], button_pos=[2.0, 5.0])

trajectory = []
costs = []

pos = aicon.agent_pos.detach().numpy()
for step in range(150):
    trajectory.append(pos)
    
    # Get optimal move directly from the math framework
    pos, cost = aicon.step(lr=0.2)
    
    costs.append(cost)
    if np.linalg.norm(pos - aicon.goal_pos.numpy()) < 0.2:
        print(f"Goal naturally reached at step {step}!")
        trajectory.append(pos)
        break

trajectory = np.array(trajectory)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Main Trajectory Plot
ax1.axvline(x=5.0, color='red', linestyle='--', alpha=0.5, label='Conditional Gate')
ax1.scatter([0], [0], color='blue', s=100, label='Start')
ax1.scatter([10], [0], color='green', s=150, marker='*', label='Goal')
ax1.scatter([2], [5], color='orange', s=120, marker='s', label='Button (Opens Gate)')

ax1.plot(trajectory[:, 0], trajectory[:, 1], 'k.-', alpha=0.5, label='AICON Generated Path')

ax1.set_xlim(-1, 11)
ax1.set_ylim(-1, 7)
ax1.set_title("AICON: Implicit Subgoal Execution without Planners")
ax1.set_xlabel("X Position")
ax1.set_ylabel("Y Position")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cost function over time plot
ax2.plot(costs, 'r-')
ax2.set_title("Differentiable Objective Function Optimization")
ax2.set_xlabel("Simulation Step")
ax2.set_ylabel("Total g(x) Cost")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Simulating the Blocks World with Matplotlib
We can also visualize the resulting plan as a simple matplotlib animation showing the implicit sequencing computed by the AICON gradients. This directly displays the results generated mathematically over the block history arrays.

In [ ]:
import sys, os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
from IPython.display import HTML

# Make sure we can import from source/
sys.path.append(os.path.abspath(os.path.join("..", "..", "source")))
from no_plan_everything_control.aicon.utils import stacks_to_on_matrix
from no_plan_everything_control.envs.blocks_world.aicon_policy import BlocksWorldAICON

# Config
NUM_BLOCKS = 5
BLOCK_SIZE = 1.0
COLORS = ['#e63946', '#2a9d8f', '#457b9d', '#f4a261', '#e9c46a']

# Initial & Goal setups
initial_stacks = [[0, 1, 2], [3], [4]]
goal_stacks = [[2, 1], [0], [4, 3]]

print(f"Goal: {goal_stacks}")
print(f"Start: {initial_stacks}")

# Setup AICON
policy = BlocksWorldAICON(n_blocks=NUM_BLOCKS, goal_on=stacks_to_on_matrix(goal_stacks, NUM_BLOCKS), interconnected_goal=True)
policy.reset(initial_stacks)
curr_stacks = [list(t) for t in initial_stacks]

# Track history of stacks
history = []
history.append([list(t) for t in curr_stacks])

# Run symbolic planner
print("Beginning symbolic AICON transitions...")
for step in range(30):
    on_mat = stacks_to_on_matrix(curr_stacks, NUM_BLOCKS)
    
    # Check goal
    if torch.allclose(on_mat, policy._goal_on):
        print(f"Goal reached at step {step}!")
        break
        
    action_tup, best_norm = policy._select_action(
        on_mat, 
        policy._state.clear, 
        policy._goal_cost(on_mat)
    )
    action, X, Y = action_tup
    
    if action == "stack":
        for t in curr_stacks:
            if X in t: t.remove(X); break
        for t in curr_stacks:
            if t and t[-1] == Y: t.append(X); break
        print(f"[{step}] Stack {X} onto {Y} -> {curr_stacks}")
    elif action == "unstack":
        for t in curr_stacks:
            if X in t: t.remove(X); break
        curr_stacks.append([X])
        print(f"[{step}] Unstack {X} from {Y} -> {curr_stacks}")
        
    # Clean up empty stacks
    curr_stacks = [t for t in curr_stacks if len(t) > 0]
    history.append([list(t) for t in curr_stacks])

# Animation
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_xlim(-1, len(history) * 1.5 + 2)
ax.set_ylim(0, NUM_BLOCKS + 1)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("AICON Blocks World Resolution", fontsize=14, pad=15)

def draw_frame(frame_idx):
    ax.clear()
    ax.set_xlim(-1, len(history[0])*1.5 + 4)
    ax.set_ylim(-0.5, NUM_BLOCKS + 1.5)
    ax.set_aspect('equal')
    ax.axis('off')
    
    ax.text(0, NUM_BLOCKS + 0.5, f"Step {frame_idx}", fontsize=12, fontweight='bold')
    
    ax.add_patch(patches.Rectangle((-1, -0.2), len(history[0])*1.5 + 4, 0.2, color='#4a4a4a'))
    
    stacks = history[frame_idx]
    
    for stack_idx, stack in enumerate(stacks):
        x = stack_idx * 1.5
        for height, block_id in enumerate(stack):
            y = height 
            rect = patches.Rectangle((x, y), BLOCK_SIZE, BLOCK_SIZE, 
                                   linewidth=2, edgecolor='black', 
                                   facecolor=COLORS[block_id % len(COLORS)],
                                   zorder=3)
            ax.add_patch(rect)
            ax.text(x + BLOCK_SIZE/2, y + BLOCK_SIZE/2, str(block_id), 
                    color='white', weight='bold', fontsize=14, 
                    ha='center', va='center', zorder=4)
                    
    ax.text(len(history[0])*1.5 + 1.5, NUM_BLOCKS - 0.5, "Goal State:", fontsize=10)
    for stack_idx, stack in enumerate(goal_stacks):
        x = (len(history[0])*1.5 + 1.5) + stack_idx * 1.2
        for height, block_id in enumerate(stack):
            y = height * 0.8
            rect = patches.Rectangle((x, y), BLOCK_SIZE*0.8, BLOCK_SIZE*0.8, 
                                   linewidth=1, edgecolor='black', 
                                   facecolor=COLORS[block_id % len(COLORS)],
                                   alpha=0.6)
            ax.add_patch(rect)
            ax.text(x + BLOCK_SIZE*0.4, y + BLOCK_SIZE*0.4, str(block_id), 
                    color='white', weight='bold', fontsize=10, 
                    ha='center', va='center')

plt.close() # Prevent static plot from showing double, we just want animation 
anim = animation.FuncAnimation(fig, draw_frame, frames=len(history), interval=800, repeat_delay=2000)

HTML(anim.to_jshtml())